# Results for model: ministral-3:14b

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import numpy as np
from datetime import datetime

def load_and_preprocess_data():
    df = pl.read_parquet('./jane-street-real-time-market-data-forecasting/train.parquet')
    df = df.with_columns(
        pl.col('date_id').cast(pl.Datetime),
        pl.col('feature_00').fill_null(0),
        pl.col('responder_6').fill_null(0)
    )
    return df

def feature_engineering(df):
    df = df.sort('date_id')
    df = df.with_columns(
        pl.col('feature_00').over('date_id').rolling_window(size=1000, min_periods=1).mean().alias('rolling_mean'),
        pl.col('feature_00').over('date_id').rolling_window(size=1000, min_periods=1).std().alias('rolling_std')
    )

    quantile_threshold = df['feature_00'].quantile(0.85, subset='date_id')
    df = df.with_columns(
        pl.when(pl.col('feature_00') >= quantile_threshold)
         .then(1)
         .otherwise(0)
         .alias('top_quantile_flag')
    )
    return df

def train_xgboost(df):
    features = ['feature_00', 'rolling_mean', 'rolling_std', 'top_quantile_flag']
    X = df.select(features).to_numpy()
    y = df['responder_6'].to_numpy()

    tscv = TimeSeriesSplit(n_splits=5)
    mse_scores = []

    for train_index, test_index in tscv.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        model = xgb.XGBRegressor(
            objective='reg:squarederror',
            n_estimators=1000,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            early_stopping_rounds=50,
            eval_metric='rmse'
        )

        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False
        )

        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        mse_scores.append(mse)

    print(f"Mean MSE across folds: {np.mean(mse_scores):.4f}")
    return model

def main():
    df = load_and_preprocess_data()
    df = feature_engineering(df)
    model = train_xgboost(df)

if __name__ == "__main__":
    main()